# 🏎️ Fabric Racing Game - Championship Edition

Race through **10 challenging tracks**, collect ⭐ **bonus stars**, avoid 🚧 **obstacles**, and climb the leaderboard!

## 🎮 Controls
- **⬆️ Arrow Up**: Accelerate
- **⬇️ Arrow Down**: Brake
- **⬅️ Arrow Left**: Steer Left
- **➡️ Arrow Right**: Steer Right
- **SPACE**: Start Race

## 📊 Scoring
- ⭐ **Star Bonus**: +100 points
- 🔥 **Consecutive Multiplier**: x2, x3, x4... (resets on miss or obstacle)
- 🚧 **Obstacle Hit**: -50 points + speed penalty
- 🏁 **Lap Bonus**: +500 points per lap
- ⏱️ **Time Bonus**: Extra points for fast completion

## 🏆 Level Progression
| Level | Track | Min Score | Stars | Obstacles |
|-------|-------|-----------|-------|----------|
| 1 | Rookie Circuit | 1,000 | 5 | 3 |
| 2 | Coastal Road | 2,000 | 7 | 5 |
| 3 | Mountain Pass | 3,500 | 8 | 7 |
| 4 | Desert Storm | 5,000 | 10 | 9 |
| 5 | Night City | 7,000 | 12 | 12 |
| 6 | Arctic Ice | 9,500 | 14 | 15 |
| 7 | Volcano Ring | 12,500 | 16 | 18 |
| 8 | Sky Highway | 16,000 | 18 | 22 |
| 9 | Quantum Loop | 20,000 | 20 | 26 |
| 10 | Champion's Road | 25,000 | 25 | 30 |

In [ ]:
# Configuration - Update these values!
EVENTSTREAM_ENDPOINT = "<YOUR_CUSTOM_ENDPOINT_URL>"
PLAYER_NAME = "Player1"  # Your name for the leaderboard

In [ ]:
from IPython.display import HTML, display
import uuid

session_id = str(uuid.uuid4())
print(f"🏎️ Game Session: {session_id[:8]}...")
print(f"👤 Player: {PLAYER_NAME}")
print(f"\n🎮 Run the next cell to start the game!")

In [ ]:
game_html = f'''
<!DOCTYPE html>
<html>
<head>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ background: #0a0a1a; font-family: 'Segoe UI', Arial, sans-serif; }}
        
        #gameContainer {{
            max-width: 900px;
            margin: 0 auto;
            padding: 10px;
        }}
        
        #header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            background: linear-gradient(135deg, #1a1a2e, #16213e);
            padding: 15px 20px;
            border-radius: 10px 10px 0 0;
            color: white;
        }}
        
        #levelInfo {{ font-size: 14px; }}
        #levelName {{ color: #00d4ff; font-weight: bold; font-size: 18px; }}
        
        #scorePanel {{
            display: flex;
            gap: 30px;
            font-size: 16px;
        }}
        .score-item {{ text-align: center; }}
        .score-label {{ color: #888; font-size: 12px; }}
        .score-value {{ color: #00ff88; font-weight: bold; font-size: 20px; }}
        #multiplier {{ color: #ff6b6b; }}
        
        #gameCanvas {{
            display: block;
            width: 100%;
            border-left: 3px solid #333;
            border-right: 3px solid #333;
        }}
        
        #footer {{
            background: #1a1a2e;
            padding: 15px;
            border-radius: 0 0 10px 10px;
            display: flex;
            justify-content: space-between;
            color: white;
        }}
        
        #progressBar {{
            width: 200px;
            height: 20px;
            background: #333;
            border-radius: 10px;
            overflow: hidden;
        }}
        #progressFill {{
            height: 100%;
            background: linear-gradient(90deg, #00d4ff, #00ff88);
            width: 0%;
            transition: width 0.3s;
        }}
        
        #leaderboard {{
            position: fixed;
            top: 10px;
            right: 10px;
            background: rgba(0,0,0,0.8);
            padding: 15px;
            border-radius: 10px;
            color: white;
            font-size: 14px;
            min-width: 200px;
            display: none;
        }}
        #leaderboard h3 {{ color: #ffd700; margin-bottom: 10px; }}
        .lb-row {{ display: flex; justify-content: space-between; padding: 3px 0; }}
        .lb-row.current {{ color: #00ff88; font-weight: bold; }}
        
        #overlay {{
            position: absolute;
            top: 0; left: 0; right: 0; bottom: 0;
            background: rgba(0,0,0,0.85);
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            color: white;
            z-index: 100;
        }}
        #overlay h1 {{ font-size: 48px; margin-bottom: 20px; }}
        #overlay p {{ font-size: 18px; margin: 10px 0; }}
        #overlay .highlight {{ color: #00ff88; font-size: 24px; }}
        .hidden {{ display: none !important; }}
        
        #levelComplete {{
            position: absolute;
            top: 50%; left: 50%;
            transform: translate(-50%, -50%);
            background: linear-gradient(135deg, #1a1a2e, #16213e);
            padding: 40px;
            border-radius: 20px;
            border: 3px solid #00ff88;
            color: white;
            text-align: center;
            z-index: 200;
        }}
        #levelComplete h2 {{ color: #ffd700; font-size: 32px; margin-bottom: 20px; }}
        #levelComplete .stats {{ font-size: 18px; margin: 15px 0; }}
        #levelComplete button {{
            background: #00ff88;
            color: #000;
            border: none;
            padding: 15px 40px;
            font-size: 18px;
            border-radius: 10px;
            cursor: pointer;
            margin-top: 20px;
        }}
        #levelComplete button:hover {{ background: #00d4ff; }}
    </style>
</head>
<body>
    <div id="gameContainer">
        <div id="header">
            <div id="levelInfo">
                <div id="levelName">Level 1: Rookie Circuit</div>
                <div>Target: <span id="targetScore">1,000</span> pts</div>
            </div>
            <div id="scorePanel">
                <div class="score-item">
                    <div class="score-label">SCORE</div>
                    <div class="score-value" id="score">0</div>
                </div>
                <div class="score-item">
                    <div class="score-label">MULTIPLIER</div>
                    <div class="score-value" id="multiplier">x1</div>
                </div>
                <div class="score-item">
                    <div class="score-label">STARS</div>
                    <div class="score-value">⭐ <span id="stars">0</span>/<span id="totalStars">5</span></div>
                </div>
                <div class="score-item">
                    <div class="score-label">LAP</div>
                    <div class="score-value" id="lap">0/3</div>
                </div>
            </div>
        </div>
        
        <div style="position: relative;">
            <canvas id="gameCanvas" width="880" height="550"></canvas>
            
            <div id="overlay">
                <h1>🏎️ FABRIC RACING</h1>
                <p class="highlight">Championship Edition</p>
                <p>10 Tracks | Collect Stars | Avoid Obstacles</p>
                <p style="margin-top: 30px;">Press <strong>SPACE</strong> to Start</p>
            </div>
            
            <div id="levelComplete" class="hidden">
                <h2>🏁 LEVEL COMPLETE!</h2>
                <div class="stats">
                    <p>Score: <span id="finalScore">0</span></p>
                    <p>Stars: <span id="finalStars">0</span></p>
                    <p>Best Combo: <span id="finalCombo">x1</span></p>
                </div>
                <p id="levelResult"></p>
                <button id="nextLevelBtn">NEXT LEVEL →</button>
            </div>
        </div>
        
        <div id="footer">
            <div>
                <span>⬆️ Accelerate</span> | 
                <span>⬇️ Brake</span> | 
                <span>⬅️➡️ Steer</span>
            </div>
            <div>
                <span>Progress to Target:</span>
                <div id="progressBar"><div id="progressFill"></div></div>
            </div>
            <div>🏆 High Score: <span id="highScore">0</span></div>
        </div>
    </div>
    
    <div id="leaderboard">
        <h3>🏆 LEADERBOARD</h3>
        <div id="lbContent"></div>
    </div>

    <script>
        const canvas = document.getElementById("gameCanvas");
        const ctx = canvas.getContext("2d");
        const ENDPOINT = "{EVENTSTREAM_ENDPOINT}";
        const PLAYER_NAME = "{PLAYER_NAME}";
        const SESSION_ID = "{session_id}";
        
        // ============== LEVEL CONFIGURATIONS ==============
        const LEVELS = [
            {{ name: "Rookie Circuit", targetScore: 1000, stars: 5, obstacles: 3, laps: 3, trackType: "oval", color: "#2d5a27" }},
            {{ name: "Coastal Road", targetScore: 2000, stars: 7, obstacles: 5, laps: 3, trackType: "wave", color: "#1a4a5e" }},
            {{ name: "Mountain Pass", targetScore: 3500, stars: 8, obstacles: 7, laps: 3, trackType: "zigzag", color: "#4a3728" }},
            {{ name: "Desert Storm", targetScore: 5000, stars: 10, obstacles: 9, laps: 4, trackType: "figure8", color: "#5a4a2a" }},
            {{ name: "Night City", targetScore: 7000, stars: 12, obstacles: 12, laps: 4, trackType: "complex", color: "#1a1a3a" }},
            {{ name: "Arctic Ice", targetScore: 9500, stars: 14, obstacles: 15, laps: 4, trackType: "slippery", color: "#3a5a6a" }},
            {{ name: "Volcano Ring", targetScore: 12500, stars: 16, obstacles: 18, laps: 5, trackType: "ring", color: "#4a2a1a" }},
            {{ name: "Sky Highway", targetScore: 16000, stars: 18, obstacles: 22, laps: 5, trackType: "aerial", color: "#2a3a5a" }},
            {{ name: "Quantum Loop", targetScore: 20000, stars: 20, obstacles: 26, laps: 5, trackType: "quantum", color: "#3a1a4a" }},
            {{ name: "Champion\'s Road", targetScore: 25000, stars: 25, obstacles: 30, laps: 6, trackType: "ultimate", color: "#1a1a1a" }}
        ];
        
        // ============== GAME STATE ==============
        let currentLevel = 0;
        let score = 0;
        let multiplier = 1;
        let maxMultiplier = 1;
        let consecutiveStars = 0;
        let starsCollected = 0;
        let gameStarted = false;
        let levelComplete = false;
        let raceStartTime = 0;
        let highScore = parseInt(localStorage.getItem("fabricRacingHighScore") || "0");
        
        // Leaderboard
        let leaderboard = JSON.parse(localStorage.getItem("fabricRacingLeaderboard") || "[]");
        
        // Car state
        let car = {{
            x: 440, y: 450,
            angle: -Math.PI/2,
            speed: 0,
            lap: 0,
            checkpoints: [false, false, false, false]
        }};
        
        // Track elements
        let stars = [];
        let obstacles = [];
        let trackPath = [];
        
        let keys = {{}};
        
        // ============== INPUT HANDLING ==============
        document.addEventListener("keydown", (e) => {{
            keys[e.key] = true;
            if (e.key === " ") {{
                if (!gameStarted && !levelComplete) startRace();
                e.preventDefault();
            }}
        }});
        document.addEventListener("keyup", (e) => keys[e.key] = false);
        
        document.getElementById("nextLevelBtn").addEventListener("click", () => {{
            if (score >= LEVELS[currentLevel].targetScore && currentLevel < 9) {{
                currentLevel++;
                initLevel();
            }} else if (currentLevel >= 9) {{
                alert("🏆 CONGRATULATIONS! You completed all 10 levels!");
            }} else {{
                initLevel(); // Retry
            }}
        }});
        
        // ============== LEVEL INITIALIZATION ==============
        function initLevel() {{
            const level = LEVELS[currentLevel];
            
            // Reset state
            score = 0;
            multiplier = 1;
            maxMultiplier = 1;
            consecutiveStars = 0;
            starsCollected = 0;
            gameStarted = false;
            levelComplete = false;
            
            // Reset car
            car = {{
                x: 440, y: 450,
                angle: -Math.PI/2,
                speed: 0,
                lap: 0,
                checkpoints: [false, false, false, false]
            }};
            
            // Generate track path based on type
            generateTrackPath(level.trackType);
            
            // Generate stars along track
            generateStars(level.stars);
            
            // Generate obstacles
            generateObstacles(level.obstacles);
            
            // Update UI
            document.getElementById("levelName").textContent = `Level ${{currentLevel + 1}}: ${{level.name}}`;
            document.getElementById("targetScore").textContent = level.targetScore.toLocaleString();
            document.getElementById("totalStars").textContent = level.stars;
            document.getElementById("highScore").textContent = highScore.toLocaleString();
            
            // Show overlay
            document.getElementById("overlay").classList.remove("hidden");
            document.getElementById("levelComplete").classList.add("hidden");
            
            updateUI();
        }}
        
        // ============== TRACK GENERATION ==============
        function generateTrackPath(type) {{
            trackPath = [];
            const cx = 440, cy = 275;
            
            switch(type) {{
                case "oval":
                    for (let i = 0; i < 360; i += 5) {{
                        const rad = i * Math.PI / 180;
                        trackPath.push({{x: cx + Math.cos(rad) * 350, y: cy + Math.sin(rad) * 200}});
                    }}
                    break;
                case "wave":
                    for (let i = 0; i < 360; i += 5) {{
                        const rad = i * Math.PI / 180;
                        const wave = Math.sin(rad * 3) * 30;
                        trackPath.push({{x: cx + Math.cos(rad) * (320 + wave), y: cy + Math.sin(rad) * (180 + wave/2)}});
                    }}
                    break;
                case "zigzag":
                    for (let i = 0; i < 360; i += 5) {{
                        const rad = i * Math.PI / 180;
                        const zig = ((i % 60) < 30 ? 1 : -1) * 25;
                        trackPath.push({{x: cx + Math.cos(rad) * (330 + zig), y: cy + Math.sin(rad) * (190 + zig/2)}});
                    }}
                    break;
                case "figure8":
                    for (let i = 0; i < 360; i += 5) {{
                        const rad = i * Math.PI / 180;
                        const r = 200 + 80 * Math.sin(rad * 2);
                        trackPath.push({{x: cx + Math.cos(rad) * r * 1.5, y: cy + Math.sin(rad) * r * 0.9}});
                    }}
                    break;
                default: // complex tracks
                    for (let i = 0; i < 360; i += 5) {{
                        const rad = i * Math.PI / 180;
                        const variation = Math.sin(rad * 4) * 40 + Math.cos(rad * 2) * 20;
                        trackPath.push({{x: cx + Math.cos(rad) * (300 + variation), y: cy + Math.sin(rad) * (170 + variation/2)}});
                    }}
            }}
        }}
        
        function generateStars(count) {{
            stars = [];
            const step = Math.floor(trackPath.length / count);
            for (let i = 0; i < count; i++) {{
                const idx = (i * step + Math.floor(step/2)) % trackPath.length;
                const pt = trackPath[idx];
                stars.push({{x: pt.x, y: pt.y, collected: false, size: 20}});
            }}
        }}
        
        function generateObstacles(count) {{
            obstacles = [];
            const step = Math.floor(trackPath.length / (count + 2));
            for (let i = 0; i < count; i++) {{
                const idx = (i * step + step) % trackPath.length;
                const pt = trackPath[idx];
                // Offset obstacles slightly from track center
                const offset = (Math.random() - 0.5) * 60;
                obstacles.push({{
                    x: pt.x + offset,
                    y: pt.y + offset * 0.5,
                    type: Math.random() > 0.5 ? "cone" : "barrier",
                    size: 25
                }});
            }}
        }}
        
        // ============== GAME LOGIC ==============
        function startRace() {{
            gameStarted = true;
            raceStartTime = Date.now();
            document.getElementById("overlay").classList.add("hidden");
            sendEvent("RaceStart");
        }}
        
        function update() {{
            if (!gameStarted || levelComplete) return;
            
            const level = LEVELS[currentLevel];
            
            // Car physics
            if (keys["ArrowUp"] || keys["w"]) car.speed = Math.min(car.speed + 0.3, 8 + currentLevel * 0.3);
            if (keys["ArrowDown"] || keys["s"]) car.speed = Math.max(car.speed - 0.4, -2);
            if (keys["ArrowLeft"] || keys["a"]) car.angle -= 0.05 * (car.speed > 0 ? 1 : -1);
            if (keys["ArrowRight"] || keys["d"]) car.angle += 0.05 * (car.speed > 0 ? 1 : -1);
            
            // Friction
            car.speed *= 0.98;
            
            // Movement
            car.x += Math.cos(car.angle) * car.speed;
            car.y += Math.sin(car.angle) * car.speed;
            
            // Boundary check
            car.x = Math.max(50, Math.min(830, car.x));
            car.y = Math.max(50, Math.min(500, car.y));
            
            // Check star collection
            stars.forEach((star, idx) => {{
                if (!star.collected) {{
                    const dist = Math.hypot(car.x - star.x, car.y - star.y);
                    if (dist < 30) {{
                        star.collected = true;
                        starsCollected++;
                        consecutiveStars++;
                        multiplier = Math.min(consecutiveStars, 10);
                        maxMultiplier = Math.max(maxMultiplier, multiplier);
                        score += 100 * multiplier;
                        sendEvent("StarCollected", {{ starIndex: idx, multiplier: multiplier }});
                    }}
                }}
            }});
            
            // Check obstacle collision
            obstacles.forEach((obs, idx) => {{
                const dist = Math.hypot(car.x - obs.x, car.y - obs.y);
                if (dist < 35) {{
                    score = Math.max(0, score - 50);
                    car.speed *= 0.3; // Speed penalty
                    consecutiveStars = 0;
                    multiplier = 1;
                    // Move obstacle slightly to prevent repeated hits
                    obs.x += (Math.random() - 0.5) * 100;
                    obs.y += (Math.random() - 0.5) * 50;
                    sendEvent("ObstacleHit", {{ obstacleIndex: idx }});
                }}
            }});
            
            // Check lap completion (simplified - crossing start line)
            checkLapProgress();
            
            // Check level completion
            if (car.lap >= level.laps) {{
                completeLap();
            }}
            
            updateUI();
        }}
        
        function checkLapProgress() {{
            // Simple checkpoint system
            const cx = 440, cy = 275;
            
            // Quadrant-based checkpoints
            if (car.x > cx && car.y < cy) car.checkpoints[0] = true;
            if (car.x < cx && car.y < cy && car.checkpoints[0]) car.checkpoints[1] = true;
            if (car.x < cx && car.y > cy && car.checkpoints[1]) car.checkpoints[2] = true;
            if (car.x > cx && car.y > cy && car.checkpoints[2]) car.checkpoints[3] = true;
            
            // Cross finish line (right side, middle)
            if (car.checkpoints[3] && car.x > 780 && car.y > 250 && car.y < 300) {{
                car.lap++;
                car.checkpoints = [false, false, false, false];
                score += 500; // Lap bonus
                sendEvent("LapComplete", {{ lap: car.lap }});
            }}
        }}
        
        function completeLap() {{
            levelComplete = true;
            const level = LEVELS[currentLevel];
            
            // Time bonus
            const raceTime = (Date.now() - raceStartTime) / 1000;
            const timeBonus = Math.max(0, Math.floor((180 - raceTime) * 10));
            score += timeBonus;
            
            // Update high score
            if (score > highScore) {{
                highScore = score;
                localStorage.setItem("fabricRacingHighScore", highScore.toString());
            }}
            
            // Update leaderboard
            updateLeaderboard();
            
            // Show completion screen
            document.getElementById("finalScore").textContent = score.toLocaleString();
            document.getElementById("finalStars").textContent = `${{starsCollected}}/${{level.stars}}`;
            document.getElementById("finalCombo").textContent = `x${{maxMultiplier}}`;
            
            const passed = score >= level.targetScore;
            const resultEl = document.getElementById("levelResult");
            const btnEl = document.getElementById("nextLevelBtn");
            
            if (passed) {{
                resultEl.innerHTML = `<span style="color:#00ff88;">✅ TARGET REACHED! +${{timeBonus}} time bonus</span>`;
                btnEl.textContent = currentLevel < 9 ? "NEXT LEVEL →" : "🏆 CHAMPION!";
            }} else {{
                resultEl.innerHTML = `<span style="color:#ff6b6b;">❌ Need ${{level.targetScore - score}} more points</span>`;
                btnEl.textContent = "RETRY LEVEL";
            }}
            
            document.getElementById("levelComplete").classList.remove("hidden");
            sendEvent("LevelComplete", {{ passed: passed, finalScore: score }});
        }}
        
        function updateLeaderboard() {{
            leaderboard.push({{ name: PLAYER_NAME, score: score, level: currentLevel + 1, time: new Date().toISOString() }});
            leaderboard.sort((a, b) => b.score - a.score);
            leaderboard = leaderboard.slice(0, 10);
            localStorage.setItem("fabricRacingLeaderboard", JSON.stringify(leaderboard));
        }}
        
        function updateUI() {{
            const level = LEVELS[currentLevel];
            document.getElementById("score").textContent = score.toLocaleString();
            document.getElementById("multiplier").textContent = `x${{multiplier}}`;
            document.getElementById("multiplier").style.color = multiplier > 1 ? "#ffd700" : "#ff6b6b";
            document.getElementById("stars").textContent = starsCollected;
            document.getElementById("lap").textContent = `${{car.lap}}/${{level.laps}}`;
            
            // Progress bar
            const progress = Math.min(100, (score / level.targetScore) * 100);
            document.getElementById("progressFill").style.width = `${{progress}}%`;
        }}
        
        // ============== RENDERING ==============
        function draw() {{
            const level = LEVELS[currentLevel];
            
            // Clear canvas
            ctx.fillStyle = level.color;
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            
            // Draw track
            ctx.strokeStyle = "#555";
            ctx.lineWidth = 80;
            ctx.lineCap = "round";
            ctx.beginPath();
            ctx.moveTo(trackPath[0].x, trackPath[0].y);
            trackPath.forEach(pt => ctx.lineTo(pt.x, pt.y));
            ctx.closePath();
            ctx.stroke();
            
            // Track lines
            ctx.strokeStyle = "#888";
            ctx.lineWidth = 2;
            ctx.setLineDash([20, 20]);
            ctx.beginPath();
            ctx.moveTo(trackPath[0].x, trackPath[0].y);
            trackPath.forEach(pt => ctx.lineTo(pt.x, pt.y));
            ctx.closePath();
            ctx.stroke();
            ctx.setLineDash([]);
            
            // Draw finish line
            ctx.fillStyle = "#fff";
            for (let i = 0; i < 8; i++) {{
                ctx.fillStyle = i % 2 === 0 ? "#fff" : "#000";
                ctx.fillRect(820, 235 + i * 10, 20, 10);
            }}
            
            // Draw obstacles
            obstacles.forEach(obs => {{
                if (obs.type === "cone") {{
                    ctx.fillStyle = "#ff6600";
                    ctx.beginPath();
                    ctx.moveTo(obs.x, obs.y - 15);
                    ctx.lineTo(obs.x - 12, obs.y + 10);
                    ctx.lineTo(obs.x + 12, obs.y + 10);
                    ctx.closePath();
                    ctx.fill();
                    ctx.strokeStyle = "#fff";
                    ctx.lineWidth = 2;
                    ctx.stroke();
                }} else {{
                    ctx.fillStyle = "#cc0000";
                    ctx.fillRect(obs.x - 15, obs.y - 8, 30, 16);
                    ctx.strokeStyle = "#fff";
                    ctx.lineWidth = 2;
                    ctx.strokeRect(obs.x - 15, obs.y - 8, 30, 16);
                }}
            }});
            
            // Draw stars
            stars.forEach(star => {{
                if (!star.collected) {{
                    ctx.save();
                    ctx.translate(star.x, star.y);
                    ctx.rotate(Date.now() / 500);
                    
                    // Glow effect
                    ctx.shadowColor = "#ffd700";
                    ctx.shadowBlur = 15;
                    
                    // Draw star
                    ctx.fillStyle = "#ffd700";
                    ctx.beginPath();
                    for (let i = 0; i < 5; i++) {{
                        const angle = (i * 4 * Math.PI / 5) - Math.PI / 2;
                        const r = i % 2 === 0 ? 12 : 6;
                        if (i === 0) ctx.moveTo(Math.cos(angle) * r, Math.sin(angle) * r);
                        else ctx.lineTo(Math.cos(angle) * r, Math.sin(angle) * r);
                    }}
                    ctx.closePath();
                    ctx.fill();
                    ctx.restore();
                }}
            }});
            
            // Draw car
            ctx.save();
            ctx.translate(car.x, car.y);
            ctx.rotate(car.angle);
            
            // Car body
            ctx.fillStyle = "#ff3333";
            ctx.beginPath();
            ctx.moveTo(20, 0);
            ctx.lineTo(-15, -12);
            ctx.lineTo(-15, 12);
            ctx.closePath();
            ctx.fill();
            
            // Cockpit
            ctx.fillStyle = "#333";
            ctx.beginPath();
            ctx.arc(0, 0, 6, 0, Math.PI * 2);
            ctx.fill();
            
            // Speed effect
            if (car.speed > 3) {{
                ctx.fillStyle = `rgba(255, 200, 0, ${{car.speed / 15}})`;
                ctx.fillRect(-30, -3, 15, 6);
            }}
            
            ctx.restore();
            
            // Multiplier display
            if (multiplier > 1) {{
                ctx.font = "bold 24px Arial";
                ctx.fillStyle = "#ffd700";
                ctx.textAlign = "center";
                ctx.fillText(`x${{multiplier}} COMBO!`, car.x, car.y - 40);
            }}
        }}
        
        // ============== EVENT STREAMING ==============
        function sendEvent(eventType, extra = {{}}) {{
            if (!ENDPOINT.startsWith("http")) return;
            
            const event = {{
                EventId: crypto.randomUUID(),
                Timestamp: new Date().toISOString(),
                SessionId: SESSION_ID,
                PlayerId: PLAYER_NAME,
                EventType: eventType,
                Level: currentLevel + 1,
                Score: score,
                Multiplier: multiplier,
                StarsCollected: starsCollected,
                Lap: car.lap,
                PositionX: Math.round(car.x),
                PositionY: Math.round(car.y),
                Speed: Math.round(car.speed * 10) / 10,
                ...extra
            }};
            
            fetch(ENDPOINT, {{
                method: "POST",
                headers: {{ "Content-Type": "application/json" }},
                body: JSON.stringify(event)
            }}).catch(e => console.log("Send error:", e));
        }}
        
        // ============== GAME LOOP ==============
        function gameLoop() {{
            update();
            draw();
            requestAnimationFrame(gameLoop);
        }}
        
        // Start
        initLevel();
        gameLoop();
        
        // Send telemetry every 500ms during race
        setInterval(() => {{
            if (gameStarted && !levelComplete) {{
                sendEvent("Telemetry");
            }}
        }}, 500);
    </script>
</body>
</html>
'''

display(HTML(game_html))

## 📊 Query Your Race Data

After playing, use these KQL queries to analyze your performance:

In [ ]:
# Sample KQL Queries (run in KQL Queryset)

kql_queries = '''
// Top scores by level
Telemetry
| where EventType == "LevelComplete"
| summarize MaxScore = max(Score) by PlayerId, Level
| order by Level asc, MaxScore desc

// Star collection efficiency
Telemetry
| where EventType == "StarCollected"
| summarize TotalStars = count(), AvgMultiplier = avg(Multiplier) by PlayerId
| order by TotalStars desc

// Obstacle hit analysis
Telemetry
| where EventType == "ObstacleHit"
| summarize Hits = count() by PlayerId, Level
| order by Hits asc

// Speed over time
Telemetry
| where EventType == "Telemetry"
| summarize AvgSpeed = avg(Speed) by bin(Timestamp, 5s), PlayerId
| render timechart
'''
print(kql_queries)